In [ ]:
import os
import numpy as np
import torch
import random
from ray import tune
from ray.rllib.algorithms.ppo import PPOConfig
from multiagent_ppo import MultiAgentF110, get_env_config, setup_policies_and_config
from ray.tune.registry import register_env
from ray.rllib.policy.policy import PolicySpec

# Set seeds for determinism
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
random.seed(SEED)

# Register the environment
register_env("f1tenth_multi", lambda config: MultiAgentF110(config))

# Create temporary environment to get spaces and agents
temp_env = MultiAgentF110(get_env_config())
policies = {agent: PolicySpec(None, temp_env.observation_space, temp_env.action_space, {}) 
            for agent in temp_env.agents}
temp_env.close()

# Configure PPO for multi-agent training with determinism
config = (PPOConfig()
          .environment("f1tenth_multi", env_config=get_env_config())
          .framework("torch")
          .api_stack(enable_rl_module_and_learner=False, enable_env_runner_and_connector_v2=False)
          .env_runners(
              num_env_runners=0, # Only use one environment runner
              num_envs_per_env_runner=1,  # Ensure single environment per worker
          )
          .multi_agent(
              policies=policies, 
              policy_mapping_fn=lambda agent_id, *args, **kwargs: agent_id
          )
          .training(
              train_batch_size=200
          )
          .evaluation(
              evaluation_interval=10,
              evaluation_num_env_runners=1,
              evaluation_config={
                  "seed": SEED + 1  # Different seed for evaluation
              }
          )
          .debugging(
              seed=SEED  # Additional seed setting
          ))

# Run training with Ray Tune
tune.run(
    "PPO",
    config=config.to_dict(),
    stop={"timesteps_total": 20000},  # Train for 20,000 timesteps
    checkpoint_freq=10,  # Save checkpoint every 10 iterations
    storage_path=os.path.abspath("./ray_results"),  # Use absolute path
    name="f1tenth_multiagent_ppo",  # Experiment name 
)

2025-06-22 23:41:45,842	INFO tune.py:616 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949


Trial name,actor_manager_num_outstanding_async_reqs,agent_timesteps_total,counters,custom_metrics,episode_media,info,num_agent_steps_sampled,num_agent_steps_sampled_lifetime,num_agent_steps_trained,num_env_steps_sampled,num_env_steps_sampled_lifetime,num_env_steps_sampled_this_iter,num_env_steps_sampled_throughput_per_sec,num_env_steps_trained,num_env_steps_trained_this_iter,num_env_steps_trained_throughput_per_sec,num_healthy_workers,num_remote_worker_restarts,num_steps_trained_this_iter,perf,timers
PPO_f1tenth_multi_b1c05_00000,0,33352,"{'num_env_steps_sampled': 19400, 'num_env_steps_trained': 19400, 'num_agent_steps_sampled': 33352, 'num_agent_steps_trained': 33352, 'num_env_steps_sampled_for_evaluation_this_iter': 2766}",{},{},"{'learner': {'agent_0': {'learner_stats': {'allreduce_latency': 0.0, 'grad_gnorm': 1.2279589494069418, 'cur_kl_coeff': 0.1623660512734205, 'cur_lr': 4.999999999999999e-05, 'total_loss': 0.17419843524694442, 'policy_loss': -0.02503071038906152, 'vf_loss': 0.19836902568737666, 'vf_explained_var': -0.010018110275268555, 'kl': 0.005297409393824637, 'entropy': 2.683705520629883, 'entropy_coeff': 0.0}, 'model': {}, 'custom_metrics': {}, 'num_agent_steps_trained': 60.0, 'num_grad_updates_lifetime': 5295.5, 'diff_num_grad_updates_vs_sampler_policy': 14.5}, 'agent_1': {'learner_stats': {'allreduce_latency': 0.0, 'grad_gnorm': 4.425850953658422, 'cur_kl_coeff': 1.5394707083702082, 'cur_lr': 5.0000000000000016e-05, 'total_loss': 0.16575770275764323, 'policy_loss': -0.013591956781844298, 'vf_loss': 0.17838610134397945, 'vf_explained_var': 0.43077598412831625, 'kl': 0.0006259017400226591, 'entropy': 3.2916111906369525, 'entropy_coeff': 0.0}, 'model': {}, 'custom_metrics': {}, 'num_agent_steps_trained': 100.0, 'num_grad_updates_lifetime': 5340.5, 'diff_num_grad_updates_vs_sampler_policy': 29.5}}, 'num_env_steps_sampled': 19400, 'num_env_steps_trained': 19400, 'num_agent_steps_sampled': 33352, 'num_agent_steps_trained': 33352, 'num_env_steps_sampled_for_evaluation_this_iter': 2766}",33352,33352,33352,19400,19400,200,198.845,19400,200,198.845,0,0,200,"{'cpu_util_percent': 59.2, 'ram_util_percent': 52.4}","{'training_iteration_time_ms': 917.842, 'restore_workers_time_ms': 0.014, 'training_step_time_ms': 917.799, 'sample_time_ms': 449.169, 'learn_time_ms': 468.483, 'learn_throughput': 426.91, 'synch_weights_time_ms': 0.005, 'restore_eval_workers_time_ms': 0.012, 'evaluation_iteration_time_ms': 4161.781, 'evaluation_iteration_throughput': 527.151}"


In [ ]:
# Visualize results
print("Training complete. Results saved to:", os.path.abspath("./ray_results"))
print("You can visualize the results using TensorBoard or Ray Dashboard.")
# Note: To visualize results, run `tensorboard --logdir=./ray_results`
# or access the Ray Dashboard at http://localhost:8265

In [2]:
from multiagent_ppo import MultiAgentF110